<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Responsible AI & Deployment (Capstone)** · Week 1 · From Notebook to Product · Lesson 2 of 24

# Reproducible Machine Learning

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/responsible-ai-deployment).

**Learning objective:** Make training repeatable with fixed seeds, configuration, pipelines, data fingerprints and pinned library versions, and prove it with a check.

## Why reproducibility matters

Six months from now someone asks: "Why did the model refuse my loan? Which data and code made it?" If you cannot rebuild the exact same model, you cannot answer, debug or defend it. **Reproducible** means: same data + same code + same settings = same model.

In [ ]:
import numpy as np
import pandas as pd

def make_applicants(n=3000, seed=7, drift=0.0, concept=0.0):
    # Made-up SACCO loan applications. drift > 0: incomes, loans and late payments change.
    # concept > 0: the link to repayment changes (salaried jobs protect less, late payments hurt more).
    rng = np.random.default_rng(seed)
    gender = rng.choice(["female", "male"], n)
    area = rng.choice(["urban", "rural"], n, p=[0.6, 0.4])
    rural = area == "rural"
    income = (rng.lognormal(12.4, 0.5, n) * np.where(rural, 0.7, 1.0) * (1 - drift * 0.25)).round(-3)
    loan = (rng.uniform(50_000, 600_000, n) * (1 + drift)).round(-3)
    years = rng.integers(0, 13, n)
    late = rng.poisson(0.7 + drift, n)
    salaried = (rng.random(n) < np.where(rural, 0.3, 0.6)).astype(int)
    momo = np.clip(rng.normal(np.where(rural, 18, 45), 10), 0, None).round().astype(int)
    score = 3.0 - 1.6 * (loan / income) + 0.12 * years - 0.7 * late + 0.6 * salaried + 0.01 * momo
    score = score - concept * (1.5 * salaried + 0.6 * late - 0.04 * (momo - 30))
    repaid = (rng.random(n) < 1 / (1 + np.exp(-score))).astype(int)
    officer = score - 0.9 * rural - 0.6 * (gender == "female") + rng.normal(0, 0.6, n)
    return pd.DataFrame({
        "applicant_id": np.arange(1, n + 1), "age": rng.integers(20, 63, n), "gender": gender, "area": area,
        "income_rwf": income, "loan_rwf": loan, "years_member": years, "late_payments": late,
        "salaried": salaried, "momo_tx_month": momo,
        "approved_before": (officer > 1.2).astype(int), "repaid": repaid})

applicants = make_applicants()
print(applicants.shape, "| repaid:", round(applicants["repaid"].mean(), 3))

## Hidden randomness

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

FEATURES = ["income_rwf", "loan_rwf", "years_member", "late_payments", "salaried", "momo_tx_month"]

def train_once(seed=None):
    tr, te = train_test_split(applicants, test_size=0.25, random_state=seed)
    forest = RandomForestClassifier(n_estimators=50, random_state=seed).fit(tr[FEATURES], tr["repaid"])
    return round(roc_auc_score(te["repaid"], forest.predict_proba(te[FEATURES])[:, 1]), 4)

print("no seed:   ", [train_once() for _ in range(3)])
print("seed = 42: ", [train_once(42) for _ in range(3)])

Without a seed, every run gives a different score — so you cannot tell whether a change helped or you were just lucky. With a fixed seed, the runs are identical.

## Put every setting in one config

In [ ]:
import hashlib
import json

CONFIG = {
    "data": {"test_size": 0.25, "split_seed": 0},
    "features": FEATURES,
    "target": "repaid",
    "model": {"type": "logistic_regression", "C": 1.0, "max_iter": 1000},
    "threshold": 0.5,
}

def config_id(config):
    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:10]

print("config id:", config_id(CONFIG))

The **config id** is a short fingerprint: if any setting changes, the id changes.

## A pipeline keeps preprocessing with the model

If you scale the data in one cell and train in another, it is easy to forget the scaling at serving time. A scikit-learn **Pipeline** stores both together, so the saved file always does the same steps:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def train(config, data):
    tr, te = train_test_split(data, test_size=config["data"]["test_size"],
                              random_state=config["data"]["split_seed"], stratify=data[config["target"]])
    pipe = Pipeline([("scale", StandardScaler()),
                     ("model", LogisticRegression(C=config["model"]["C"], max_iter=config["model"]["max_iter"]))])
    pipe.fit(tr[config["features"]], tr[config["target"]])
    p = pipe.predict_proba(te[config["features"]])[:, 1]
    metrics = {"auc": round(roc_auc_score(te[config["target"]], p), 4),
               "accuracy": round(accuracy_score(te[config["target"]], p >= config["threshold"]), 4)}
    return pipe, metrics

_, first = train(CONFIG, applicants)
_, second = train(CONFIG, applicants)
print(first, second)
assert first == second, "training is not reproducible!"
print("reproducible: the same config and data give the same model")

That `assert` is a tiny **reproducibility test**. Put it in your test suite (Lesson 4).

## Fingerprint the data

In [ ]:
def data_fingerprint(df):
    return hashlib.sha256(pd.util.hash_pandas_object(df, index=False).values.tobytes()).hexdigest()[:16]

print("original data:", data_fingerprint(applicants))
changed = applicants.copy()
changed.loc[0, "income_rwf"] += 1000          # change ONE number
print("one value changed:", data_fingerprint(changed))

One changed value gives a completely different fingerprint. Store the fingerprint with every model — and keep the data itself in a safe, versioned place (a dated folder, object storage, or tools such as DVC).

## Pin your library versions

In [ ]:
import importlib.metadata as md

packages = ["scikit-learn", "pandas", "numpy", "joblib"]
pinned = [f"{p}=={md.version(p)}" for p in packages]
with open("requirements.txt", "w") as f:
    f.write("\n".join(pinned) + "\n")
print(open("requirements.txt").read())

`requirements.txt` with exact versions (`==`) lets anyone install the same libraries later.

## A tidy folder layout

```text
loan-model/
  data/            raw and processed data (or links to where it is stored)
  src/             train.py, features.py, serve.py
  tests/           test_data.py, test_model.py, test_api.py
  configs/         train.json
  models/          saved models + metadata (or a model registry)
  requirements.txt
  README.md        how to train, test and run
```

## Key terms

- **Reproducible** — the same inputs always give the same model and results.
- **Random seed** — a number that fixes "random" choices.
- **Config** — one place that holds every training setting.
- **Pipeline** — preprocessing and model saved together.
- **Data fingerprint** — a hash that identifies the exact data.
- **Pinned versions** — exact library versions in requirements.txt.

**Takeaway:** Fix seeds, keep settings in a config, use pipelines, fingerprint the data and pin library versions — then prove reproducibility with a test.

## Try it yourself

1. Change `C` to 0.1 in `CONFIG`. Does the config id change? Does the AUC change?
2. Remove `stratify` from `train`. Is it still reproducible?
3. Write a function `save_run(config, metrics, data)` that stores the config, metrics and data fingerprint in one JSON file.
4. Create the folder layout above for one of your own projects.

---
Finished? Go back to [Responsible AI & Deployment (Capstone) on WAMUGII](https://wamugii.kento.rw/courses/responsible-ai-deployment), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*